In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import os
import time
import numpy as np
import cv2
import seaborn as sns
import keras
import keras_tuner as kt

from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from tensorflow.keras.layers import Rescaling
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras import applications, models, layers, Sequential
from keras.optimizers import AdamW
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.models import load_model
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
TRAIN_DATA_PATH = "../affectnet_dataset/Train"
TEST_DATA_PATH = "../affectnet_dataset/Test"
EPOCHS = 100
RANDOM_SEED = 40
BATCH_SIZE = 32
IMG_SIZE = (96,96)

SAVED_MODEL = "transfer_model.h5"

In [ ]:
# flow_from_directory is older approach - slower
train_dataset = image_dataset_from_directory(
    TRAIN_DATA_PATH,
    validation_split=0.2,
    subset="training",
    seed = RANDOM_SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode = "int"
)

val_dataset = image_dataset_from_directory(
    TRAIN_DATA_PATH,
    validation_split=0.2,
    seed = RANDOM_SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode = "int",
    subset="validation"
)

test_dataset = image_dataset_from_directory(
    TEST_DATA_PATH,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode = "int"
)

In [ ]:
train_counts = {}
CLASSES = train_dataset.class_names

for cls in CLASSES:
    cls_folder = os.path.join(TRAIN_DATA_PATH, cls)
    train_counts[cls] = len(os.listdir(cls_folder))

print(train_counts)

In [ ]:
train_dataset = train_dataset.map(
    lambda x, y: (preprocess_input(x), y)
)

val_dataset = val_dataset.map(
    lambda x, y: (preprocess_input(x), y)
)

test_dataset = test_dataset.map(
    lambda x, y: (preprocess_input(x), y)
)

In [ ]:
print(train_dataset.class_names)

In [ ]:
'''
Calculate class weights for imbalanced dataset
Create label list manually from folder counts
Class weights змушують loss сильніше штрафувати помилки на rare classes.
'''
labels = []

for idx, cls in enumerate(CLASSES):
    count = train_counts[cls]
    labels.extend([idx] * count)

labels = np.array(labels)

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights = dict(
    zip(range(len(class_weights_array)), class_weights_array)
)
print("Class weights:", class_weights)

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05), # 0.1
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1)
], name="data_augmentation")

In [ ]:
base_model = applications.MobileNetV2(
    input_shape=(*IMG_SIZE, 3),
    include_top=False, # rm last classification block
    weights = "imagenet" # take weights pretrained on imagenet
)
# Freeze base block
base_model.trainable = False

In [ ]:
# model = tf.keras.Sequential([
#     layers.Input(shape=(*IMG_SIZE, 3)),
#     data_augmentation,
#     # preprocess input (px range [-1, 1]) for mobilenet
#     # generally preprocess rescale = 1./255 => [0,1]
#     base_model,
#     # decrease to vector with avg
#     layers.GlobalAveragePooling2D(),

#     layers.Dense(
#         256,
#         kernel_regularizer=tf.keras.regularizers.l2(1e-4)
#     ),
#     layers.BatchNormalization(),
#     layers.Activation('relu'),
#     layers.Dropout(0.4),
    
#     layers.Dense(128,
#         kernel_regularizer=tf.keras.regularizers.l2(1e-4)
#     ),
#     layers.BatchNormalization(),
#     layers.ReLU,
#     layers.Dropout(0.3),
    
#     layers.Dense(8,activation='softmax')
# ])
# # sparse_categorical_crossentropy - labels are numbers
# # categorical_crossentropy - labels are 1-hot encoding (label_mode="categorical" || tf.one_hot())
# model.compile(loss='sparse_categorical_crossentropy', optimizer=AdamW(learning_rate=1e-4, weight_decay=1e-5), metrics=['accuracy'])

# # Adam - (w - lr*gradient)
# # AdamW - Adam + weight decay(regularization, avoid exploiding weights) w - lr*gradient - w_d*w (model compresses large w)
# model.summary()

In [ ]:
def build_model(hp):
    model = tf.keras.Sequential([
        layers.Input(shape=(*IMG_SIZE, 3)),
        data_augmentation,
        base_model,

        layers.GlobalAveragePooling2D(),

        layers.Dense(
            256,
            kernel_regularizer=tf.keras.regularizers.l2(
                hp.Float('l2_1', 1e-5, 1e-3, sampling='log')
            )
        ),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Dropout(hp.Float('dropout4',min_value=0.1, max_value=0.5, step=0.1)),
        
        layers.Dense(128,
            kernel_regularizer=tf.keras.regularizers.l2(
                hp.Float('l2_2', 1e-5, 1e-3, sampling='log')
            )
        ),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Dropout(hp.Float('dropout5',min_value=0.1, max_value=0.5, step=0.1)),
        layers.Dense(8,activation='softmax')
    ])
    model.compile(
        loss='sparse_categorical_crossentropy',
        optimizer=AdamW(hp.Float('lr',min_value=1e-5, max_value=1e-3, sampling='log')),
        metrics=['accuracy']
    )
    return model

In [ ]:
'''
Initialize a tuner (here, HyperBand). 
We use objective to specify the objective to select the best models,
and we use max_trials to specify the number of different models to try.
'''
tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=20,
    factor=3, # Reduction factor 3x fewer models in the next round
    directory='keras_tuner_results',
    project_name='emotion_recognition'
)

In [ ]:
tuner.search_space_summary()

In [ ]:
tuner.search(
    train_dataset,
    epochs=10,
    validation_data=val_dataset
)

In [ ]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
best_model = build_model(best_hp)
best_model.summary()

In [ ]:
best_hp.values

In [ ]:
tuner.results_summary()

In [ ]:
es = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)
# es monitored val_loss
mc = ModelCheckpoint(SAVED_MODEL, monitor='val_accuracy', save_best_only=True)
csv_logger = CSVLogger('cnn_weighted_loss_training_log.csv')
rlr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5, # 0.3
    patience=2,  #3  
    min_lr=1e-6
)

In [ ]:
start = time.time()
history = best_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights,
    callbacks=[es, mc, csv_logger, rlr]
)
end = time.time()
elapsed_time = end - start

print("Training time: ", time.strftime("%H:%M:%S", time.gmtime(elapsed_time)))

In [ ]:
plt.figure(figsize=(12,5))

# Loss
plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Accuracy
plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

In [ ]:
best_model = tf.keras.models.load_model(SAVED_MODEL)

In [ ]:
test_loss, test_acc = best_model.evaluate(test_dataset)
print(f"Test Accuracy: {test_acc*100:.2f}%")

In [ ]:
print("Evaluating on Test Set...")

loss, accuracy = model.evaluate(test_dataset)

print(f"Test Accuracy: {accuracy*100:.2f}%")

print("Generating predictions...")

predictions = model.predict(test_dataset, verbose=1)

# Predicted classes
y_pred_indices = np.argmax(predictions, axis=1)

# True labels
y_true_indices = np.concatenate([
    y.numpy() for x, y in test_dataset
])

# Class names
class_labels = CLASSES

print(len(y_true_indices))
print(len(y_pred_indices))

print("\nClassification Report:\n")

ls = list(range(len(CLASSES)))

print(classification_report(
    y_true_indices,
    y_pred_indices,
    labels=labels,
    target_names=CLASSES,
    zero_division=0
))

cm = confusion_matrix(y_true_indices, y_pred_indices, labels=labels)

# Plot
plt.figure(figsize=(10, 8))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_labels,
    yticklabels=class_labels
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")

plt.tight_layout()

plt.show()

In [ ]:
CLASSES = list(test_dataset.class_indices.keys())
CLASSES

In [ ]:
predictions = best_model.predict(test_dataset, verbose=1)

# Convert predictions to class indexes
y_pred_indices = np.argmax(predictions, axis=1)

# Get true labels directly from the generator
y_true_indices = test_dataset.classes

# axis 0 and axis 1 depend on the shape of predictions (num_samples, num_classes)
confidence = np.max(predictions, axis=1)
print("Confidence scores for predictions: ", confidence)

wrong_idx = np.where(y_pred_indices != y_true_indices)[0]
print("Total wrong predictions: ", len(wrong_idx))

high_confidence_wrong_idx = wrong_idx[confidence[wrong_idx] > 0.8]
print("High confidence wrong predictions: ", len(high_confidence_wrong_idx))

In [ ]:
def predict_random_samples():

    # Get one batch
    images, labels = next(iter(test_dataset))

    # Random 5 images
    indices = np.random.choice(len(images), 5, replace=False)

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))

    for i, idx in enumerate(indices):

        img = images[i]
        # True label
        true_idx = int(labels[idx])
        true_label = CLASSES[true_idx]

        # Prediction
        pred_prob = best_model.predict(
            np.expand_dims(img, axis=0),
            verbose=0
        )

        pred_idx = np.argmax(pred_prob)
        pred_label = CLASSES[pred_idx]

        # Plot image
        axes[i].imshow((img * 255).astype(np.uint8))
        axes[i].axis("off")

        # Color
        color = 'green' if true_idx == pred_idx else 'red'

        axes[i].set_title(
            f"True: {true_label}\nPred: {pred_label}",
            color=color
        )

    plt.tight_layout()
    plt.show()

predict_random_samples()